# Feature Engineering & Sequence Generation (FD001 + FD004)

## Purpose
Prepare multi-condition C-MAPSS data for deep learning (LSTM/GRU) by:
1. Loading labeled datasets
2. Creating robust features across operating regimes (FD004)
3. Scaling features safely (no data leakage)
4. Generating time-series windows for sequence models
5. Building training/validation/test arrays: (X, y)

## Aviation-style approach
FD004 contains multiple operating conditions and fault modes. We reduce condition-driven noise by:
- normalizing sensors by operating regime (coarse binning)
- using rolling statistics to capture degradation trend
- applying safety-oriented targets (RUL capping)

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib

# Absolute base directory (stable)
BASE_DIR = Path(r"C:\Users\Kal\aircraft-engine-safety-risk-prediction")

PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

fd001_path = PROCESSED_DIR / "fd001_train_labeled.csv"
fd004_path = PROCESSED_DIR / "fd004_train_labeled.csv"
fd004_test_path = PROCESSED_DIR / "fd004_test_labeled.csv"

print("FD001:", fd001_path.exists(), fd001_path)
print("FD004:", fd004_path.exists(), fd004_path)
print("FD004 TEST:", fd004_test_path.exists(), fd004_test_path)

FD001: True C:\Users\Kal\aircraft-engine-safety-risk-prediction\data\processed\fd001_train_labeled.csv
FD004: True C:\Users\Kal\aircraft-engine-safety-risk-prediction\data\processed\fd004_train_labeled.csv
FD004 TEST: True C:\Users\Kal\aircraft-engine-safety-risk-prediction\data\processed\fd004_test_labeled.csv


## 1) Load Data
We load labeled training sets (FD001, FD004) and labeled FD004 test set.

Important:
- We keep test set completely unseen during scaling and feature selection.

In [9]:
fd001 = pd.read_csv(fd001_path)
fd004 = pd.read_csv(fd004_path)
fd004_test = pd.read_csv(fd004_test_path)

print(fd001.shape, fd004.shape, fd004_test.shape)
fd004.head()

(20631, 27) (61249, 27) (41214, 27)


,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,RUL
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670,320
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552,319
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213,318
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176,317
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754,316


## Sensor Selection

Based on variance and RUL correlation screening from Notebook 1, we remove low-information sensors before feature engineering.

These sensors showed near-constant behavior or weak predictive signal across FD001 and FD004.

In [10]:
# Start with all sensor columns
sensor_cols = [c for c in fd004.columns if c.startswith("s")]

# Low-information sensors identified from Notebook 1
drop_sensors = ["s1", "s5", "s6", "s10", "s16", "s18", "s19"]

# Keep only useful sensors
sensor_cols = [c for c in sensor_cols if c not in drop_sensors]

print("Selected sensor columns:")
print(sensor_cols)
print("Number of selected sensors:", len(sensor_cols))

Selected sensor columns:
['s2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']
Number of selected sensors: 14


## 2) RUL Capping (Safety-Style Target)
In many predictive maintenance standards, RUL is capped to reduce extreme targets and improve stability.
This also supports safety classification later.

Common cap values: 125 or 150 cycles.
We use 125 (standard baseline in many C-MAPSS tutorials).

In [11]:
RUL_CAP = 125

for df in [fd001, fd004, fd004_test]:
    df["RUL_capped"] = df["RUL"].clip(upper=RUL_CAP)

fd004[["RUL", "RUL_capped"]].describe()

,RUL,RUL_capped
count,61249.000000,61249.000000
mean,133.311417,92.985192
std,89.783389,40.665112
min,0.000000,0.000000
25%,61.000000,61.000000
50%,122.000000,122.000000
75%,190.000000,125.000000
max,542.000000,125.000000


## 3) Operating Regime Normalization (FD004 Robustness)
FD004 includes multiple operating conditions. Sensors can shift due to conditions rather than degradation.

We create coarse regime bins from operating settings and normalize sensors within each regime.
This reduces condition-driven variance and improves generalization.

In [12]:
# Keep only selected useful sensors
sensor_cols = ['s2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']
op_cols = ["op1", "op2", "op3"]

def add_regime(df: pd.DataFrame, bins=4) -> pd.DataFrame:
    out = df.copy()
    out["op1_bin"] = pd.qcut(out["op1"], q=bins, duplicates="drop")
    out["op2_bin"] = pd.qcut(out["op2"], q=bins, duplicates="drop")
    out["op3_bin"] = pd.qcut(out["op3"], q=bins, duplicates="drop")
    out["regime"] = (
        out["op1_bin"].astype(str)
        + "|"
        + out["op2_bin"].astype(str)
        + "|"
        + out["op3_bin"].astype(str)
    )
    return out

def regime_normalize(train_df: pd.DataFrame, apply_df: pd.DataFrame, sensors: list[str]) -> pd.DataFrame:
    stats = train_df.groupby("regime")[sensors].agg(["mean", "std"])
    stats.columns = [f"{c[0]}_{c[1]}" for c in stats.columns]

    def transform(df):
        df = df.copy()
        df = df.merge(stats, left_on="regime", right_index=True, how="left")

        for s in sensors:
            mu = df[f"{s}_mean"]
            sd = df[f"{s}_std"].replace(0, np.nan)
            df[s] = (df[s] - mu) / sd

        drop_cols = [c for c in df.columns if c.endswith("_mean") or c.endswith("_std")]
        df = df.drop(columns=drop_cols)

        return df

    return transform(apply_df)

# Add regimes (FD004 only)
fd004_r = add_regime(fd004, bins=4)
fd004_test_r = add_regime(fd004_test, bins=4)

# Keep clean train copy for regime stats
fd004_train_regime = fd004_r.copy()

# Normalize using TRAIN statistics only
fd004_r = regime_normalize(fd004_train_regime, fd004_r, sensor_cols)
fd004_test_r = regime_normalize(fd004_train_regime, fd004_test_r, sensor_cols)

# Fill missing values safely
fd004_r[sensor_cols] = fd004_r[sensor_cols].fillna(0)
fd004_test_r[sensor_cols] = fd004_test_r[sensor_cols].fillna(0)

# Optional but recommended: clip extreme normalized values
fd004_r[sensor_cols] = fd004_r[sensor_cols].clip(-5, 5)
fd004_test_r[sensor_cols] = fd004_test_r[sensor_cols].clip(-5, 5)

# FD001 has one operating condition, keep as-is for now
fd001_r = fd001.copy()

print("Selected sensors:", sensor_cols)
print("Number of selected sensors:", len(sensor_cols))
print("FD004 train NaNs:", fd004_r[sensor_cols].isna().sum().sum())
print("FD004 test NaNs :", fd004_test_r[sensor_cols].isna().sum().sum())
print("FD004 train max :", fd004_r[sensor_cols].max().max())
print("FD004 test max  :", fd004_test_r[sensor_cols].max().max())
print("FD004 train min :", fd004_r[sensor_cols].min().min())
print("FD004 test min  :", fd004_test_r[sensor_cols].min().min())

fd004_r[["regime"] + sensor_cols[:3]].head()

Selected sensors: ['s2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']
Number of selected sensors: 14
FD004 train NaNs: 0
FD004 test NaNs : 0
FD004 train max : 5.0
FD004 test max  : 4.661232339540687
FD004 train min : -5.0
FD004 test min  : -5.0


,regime,s2,s3,s4
0,"(41.998, 42.008]|(0.7, 0.84]|(59.999, 100.0]",0.427922,-1.382989,-1.283267
1,"(10.005, 25.001]|(0.7, 0.84]|(59.999, 100.0]",-2.480739,-0.884002,-1.359522
2,"(41.998, 42.008]|(0.84, 0.842]|(59.999, 100.0]",-1.144623,-1.435548,-0.823650
3,"(41.998, 42.008]|(0.7, 0.84]|(59.999, 100.0]",-1.666594,-1.714562,-0.719610
4,"(25.001, 41.998]|(0.251, 0.7]|(59.999, 100.0]",-1.660600,-0.934239,-1.778752


## 4) Rolling Trend Features (Degradation Signals)
We compute rolling statistics per engine:
- rolling mean (smooth)
- rolling std (variability)

This helps the model capture degradation trend, not noise.

In [13]:
def add_rolling_features(df: pd.DataFrame, sensors: list[str], window=5) -> pd.DataFrame:
    out = df.sort_values(["unit", "cycle"]).copy()

    for s in sensors:
        out[f"{s}_rollmean"] = (
            out.groupby("unit")[s]
            .transform(lambda x: x.rolling(window, min_periods=1).mean())
        )

        out[f"{s}_rollstd"] = (
            out.groupby("unit")[s]
            .transform(lambda x: x.rolling(window, min_periods=1).std())
            .fillna(0.0)
        )

    return out


ROLL_WINDOW = 5

# Add rolling features using selected sensors only
fd001_r = add_rolling_features(fd001_r, sensor_cols, window=ROLL_WINDOW)
fd004_r = add_rolling_features(fd004_r, sensor_cols, window=ROLL_WINDOW)
fd004_test_r = add_rolling_features(fd004_test_r, sensor_cols, window=ROLL_WINDOW)

# Final feature list: raw sensors + rolling mean + rolling std
feature_cols = (
    sensor_cols
    + [f"{s}_rollmean" for s in sensor_cols]
    + [f"{s}_rollstd" for s in sensor_cols]
)

print("Total features:", len(feature_cols))
print("First 10 features:", feature_cols[:10])

Total features: 42
First 10 features: ['s2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14']


## 5) Global Scaling (No Data Leakage)
We fit the scaler on TRAIN ONLY (FD001 + FD004 training).
Then apply to validation/test.

This ensures a realistic evaluation.

In [14]:
# Combine training for fitting scaler
train_all = pd.concat([fd001_r, fd004_r], axis=0, ignore_index=True)

scaler = StandardScaler()
scaler.fit(train_all[feature_cols])

# Apply
fd001_r[feature_cols] = scaler.transform(fd001_r[feature_cols])
fd004_r[feature_cols] = scaler.transform(fd004_r[feature_cols])
fd004_test_r[feature_cols] = scaler.transform(fd004_test_r[feature_cols])

joblib.dump(scaler, MODELS_DIR / "scaler.pkl")
print("Saved scaler:", (MODELS_DIR / "scaler.pkl").resolve())

Saved scaler: C:\Users\Kal\aircraft-engine-safety-risk-prediction\models\scaler.pkl


## 6) Sequence Generation (LSTM/GRU Input)
We create sliding windows per engine:
- X shape: (num_samples, window, num_features)
- y: RUL_capped at the end of each window

Aviation-style choice:
- Use a moderate window size (e.g., 30 cycles) to capture trend.

In [15]:
import numpy as np
import pandas as pd

def make_sequences(df: pd.DataFrame,
                   features: list[str],
                   target_col="RUL_capped",
                   window=30,
                   stride=1):
    """
    Create sliding windows per engine (unit) and return:
    X: sequences (num_samples, window, num_features)
    y: target at end of each window (num_samples,)
    units: engine ids for each window (num_samples,)
    end_cycles: cycle number at end of each window (num_samples,)
    """
    df = df.sort_values(["unit", "cycle"]).copy()

    X_list, y_list, unit_list, end_cycle_list = [], [], [], []

    for uid, g in df.groupby("unit"):
        g = g.reset_index(drop=True)

        data = g[features].values
        target = g[target_col].values
        cycles = g["cycle"].values

        for start in range(0, len(g) - window + 1, stride):
            end = start + window

            X_list.append(data[start:end])
            y_list.append(target[end - 1])
            unit_list.append(uid)
            end_cycle_list.append(cycles[end - 1])

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    units = np.array(unit_list)
    end_cycles = np.array(end_cycle_list)

    return X, y, units, end_cycles


def select_last_window_per_engine(X, y, units, end_cycles):
    """
    Airline-style evaluation: select the last (latest) window per engine.
    Returns one sample per engine.
    """
    idx_df = pd.DataFrame({
        "idx": np.arange(len(y)),
        "unit": units,
        "end_cycle": end_cycles
    })

    last_idx = (
        idx_df.sort_values(["unit", "end_cycle"])
              .groupby("unit")
              .tail(1)["idx"]
              .values
    )

    return X[last_idx], y[last_idx], units[last_idx], end_cycles[last_idx]


# Parameters
WINDOW = 30
STRIDE = 1

# Generate sequences
X_fd001, y_fd001, u_fd001, c_fd001 = make_sequences(
    fd001_r, feature_cols, target_col="RUL_capped", window=WINDOW, stride=STRIDE
)

X_fd004, y_fd004, u_fd004, c_fd004 = make_sequences(
    fd004_r, feature_cols, target_col="RUL_capped", window=WINDOW, stride=STRIDE
)

X_test, y_test, u_test, c_test = make_sequences(
    fd004_test_r, feature_cols, target_col="RUL_capped", window=WINDOW, stride=STRIDE
)

print("FD001 sequences:", X_fd001.shape, y_fd001.shape)
print("FD004 sequences:", X_fd004.shape, y_fd004.shape)
print("FD004 TEST seqs:", X_test.shape, y_test.shape)

print("\nSequence feature dimension:")
print("FD001 feature count:", X_fd001.shape[2])
print("FD004 feature count:", X_fd004.shape[2])
print("FD004 TEST feature count:", X_test.shape[2])

# Airline-style: last window per engine
X_test_last, y_test_last, u_test_last, c_test_last = select_last_window_per_engine(
    X_test, y_test, u_test, c_test
)

print("\nAA-style evaluation set (last window per engine):")
print("FD004 TEST last-window:", X_test_last.shape, y_test_last.shape)
print("Unique engines in test:", len(np.unique(u_test_last)))

FD001 sequences: (17731, 30, 42) (17731,)
FD004 sequences: (54028, 30, 42) (54028,)
FD004 TEST seqs: (34081, 30, 42) (34081,)

Sequence feature dimension:
FD001 feature count: 42
FD004 feature count: 42
FD004 TEST feature count: 42

AA-style evaluation set (last window per engine):
FD004 TEST last-window: (237, 30, 42) (237,)
Unique engines in test: 237


## 7) Create Validation Split (FD004)
We use FD004 as the main robustness dataset.
We split engines (not rows) to avoid leakage.

Validation engines are held out entirely.

In [16]:
from sklearn.model_selection import train_test_split

# Get unique engine IDs from FD004
unique_units = np.unique(u_fd004)

# Split engines into train and validation
train_units, val_units = train_test_split(
    unique_units,
    test_size=0.2,
    random_state=42
)

# Create masks based on engine IDs
train_mask = np.isin(u_fd004, train_units)
val_mask   = np.isin(u_fd004, val_units)

# Apply masks
X_fd004_train = X_fd004[train_mask]
y_fd004_train = y_fd004[train_mask]

X_fd004_val = X_fd004[val_mask]
y_fd004_val = y_fd004[val_mask]

print("FD004 train:", X_fd004_train.shape, y_fd004_train.shape)
print("FD004 val  :", X_fd004_val.shape, y_fd004_val.shape)

FD004 train: (43523, 30, 42) (43523,)
FD004 val  : (10505, 30, 42) (10505,)


## 8) Save Arrays (Optional)
We save arrays so model training notebook can load quickly.

In [27]:
OUT_DIR = BASE_DIR / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save FD001
np.save(OUT_DIR / "X_fd001.npy", X_fd001)
np.save(OUT_DIR / "y_fd001.npy", y_fd001)

# Save FD004 train / validation
np.save(OUT_DIR / "X_fd004_train.npy", X_fd004_train)
np.save(OUT_DIR / "y_fd004_train.npy", y_fd004_train)

np.save(OUT_DIR / "X_fd004_val.npy", X_fd004_val)
np.save(OUT_DIR / "y_fd004_val.npy", y_fd004_val)

# Save FD004 test (all windows)
np.save(OUT_DIR / "X_fd004_test.npy", X_test)
np.save(OUT_DIR / "y_fd004_test.npy", y_test)

# Save FD004 test (last window per engine - airline style)
np.save(OUT_DIR / "X_fd004_test_last.npy", X_test_last)
np.save(OUT_DIR / "y_fd004_test_last.npy", y_test_last)
np.save(OUT_DIR / "u_fd004_test_last.npy", u_test_last)
np.save(OUT_DIR / "c_fd004_test_last.npy", c_test_last)

print("Saved arrays to:", OUT_DIR.resolve())

Saved arrays to: C:\Users\Kal\aircraft-engine-safety-risk-prediction\data\processed


# Summary

Completed:
- RUL capping (stable safety-style target)
- Operating regime normalization (FD004 robustness)
- Rolling trend features (degradation signal)
- Scaling without leakage
- Sequence generation for LSTM/GRU
- Engine-level validation split

Next:
Train LSTM/GRU models and evaluate MAE/RMSE on FD004 validation and FD004 test.